# 6. Arrays and DataFrames

배열의 axis, 표의 label, join의 key를 추적하여 오류 없이 실행되지만 뜻이 틀린 분석을 찾습니다.

핵심 셀은 NumPy와 pandas를 사용합니다. 유료 API, GPU, 외부 데이터 다운로드는 필요하지 않습니다.

## 1. Axis와 Broadcasting

행은 item, 열은 서로 다른 feature입니다. 서로 다른 단위의 feature를 무작정 평균하지 않습니다.

In [1]:
import numpy as np
import pandas as pd
X=np.array([[10,100],[20,200],[30,300]],dtype=float)
print("shape",X.shape,"axis0",X.mean(axis=0),"axis1",X.mean(axis=1))
assert np.allclose(X.mean(axis=0),[20,200])
print("Feature offsets:\n",X+np.array([1,2]))
print("Row offsets:\n",X+np.array([1,2,3])[:,None])

shape (3, 2) axis0 [ 20. 200.] axis1 [ 55. 110. 165.]
Feature offsets:
 [[ 11. 102.]
 [ 21. 202.]
 [ 31. 302.]]
Row offsets:
 [[ 11. 101.]
 [ 22. 202.]
 [ 33. 303.]]


## 2. View의 공유 저장소

NumPy basic slice는 원본의 view일 수 있습니다. 원본을 보존해야 하면 copy를 명시합니다.

In [2]:
original=np.array([1,2,3,4])
view=original[:2]
view[0]=99
assert original[0]==99
copy=original[:2].copy()
copy[0]=0
assert original[0]==99
print(original,copy)

[99  2  3  4] [0 2]


## 3. Labelled Table과 Group 단위

aggregation은 group당 한 행, transform은 원래 관측당 한 행입니다.

In [3]:
items=pd.DataFrame({"item":["p03","p08","p11"],"line":["A","A","B"],"length":[10,20,30]})
group_mean=items.groupby("line")["length"].mean()
items["line_mean"]=items.groupby("line")["length"].transform("mean")
print(group_mean.to_dict())
print(items.to_string(index=False))
assert len(group_mean)==2 and len(items)==3
assert items["line_mean"].tolist()==[15.,15.,30.]

{'A': 15.0, 'B': 30.0}
item line  length  line_mean
 p03    A      10       15.0
 p08    A      20       15.0
 p11    B      30       30.0


## 4. Join의 Cardinality

lookup key가 중복이면 item 행이 늘어날 수 있습니다. validate를 빼고 나온 결과와 비교하세요.

In [4]:
lookup=pd.DataFrame({"line":["A","B"],"region":["North","South"]})
joined=items.merge(lookup,on="line",how="left",validate="many_to_one",indicator=True)
assert len(joined)==len(items) and joined["region"].notna().all()
print(joined[["item","line","region","_merge"]].to_string(index=False))
bad_lookup=pd.concat([lookup,lookup.iloc[[0]]],ignore_index=True)
unchecked=items.merge(bad_lookup,on="line",how="left")
print("Unchecked rows:",len(unchecked))
try: items.merge(bad_lookup,on="line",validate="many_to_one")
except pd.errors.MergeError: print("Duplicate lookup key rejected")
else: raise AssertionError("duplicate lookup key was accepted")

item line region _merge
 p03    A  North   both
 p08    A  North   both
 p11    B  South   both
Unchecked rows: 5
Duplicate lookup key rejected


## 5. 과제와 해답

- lookup에서 Line B를 제거하고 행 수와 미매칭을 모두 검사하세요.
- 두 feature의 scale을 바꿔 nearest neighbor가 달라지는 입력을 계산하세요.

In [5]:
unmatched=items.merge(lookup[lookup.line=="A"],on="line",how="left",validate="many_to_one",indicator=True)
assert len(unmatched)==3 and (unmatched["_merge"]=="left_only").sum()==1
print(unmatched[["item","region","_merge"]].to_string(index=False))

item region    _merge
 p03  North      both
 p08  North      both
 p11    NaN left_only


In [6]:
points=np.array([[0,0],[1,1],[4,0],[4,3]],dtype=float)
query=np.array([1,0],dtype=float)
for scale in [1.,3.,.25]:
    distance=np.linalg.norm((points-query)*[scale,1],axis=1)
    nearest=np.flatnonzero(np.isclose(distance,distance.min()))
    print("scale",scale,"distances",distance.round(3),"nearest",nearest)
assert np.allclose(np.linalg.norm(points-query,axis=1),[1,1,3,np.sqrt(18)])

scale 1.0 distances [1.    1.    3.    4.243] nearest [0 1]
scale 3.0 distances [3.    1.    9.    9.487] nearest [1]
scale 0.25 distances [0.25  1.    0.75  3.092] nearest [0]


## 6. 참고: Alignment, Window, Pivot

label과 시간을 보고 결과의 단위를 설명하세요. 아래 window는 과거 두 행을 사용합니다. 미래 prediction용 feature라면 현재 target 포함 여부도 별도로 정해야 합니다.

In [7]:
left=pd.Series([10,20],index=["a","b"])
right=pd.Series([1,2],index=["a","c"])
print("Aligned sum:", (left+right).to_dict())
values=pd.Series([10,20,30,40])
print("Rolling:",values.rolling(2).mean().tolist())
long=pd.DataFrame({"item":["A","A","B"],"day":[1,2,1],"value":[10,12,20]})
print(long.pivot(index="item",columns="day",values="value"))

Aligned sum: {'a': 11.0, 'b': nan, 'c': nan}
Rolling: [nan, 15.0, 25.0, 35.0]
day      1     2
item            
A     10.0  12.0
B     20.0   NaN
